In [1]:
%load_ext sql
%sql sqlite:///ma_base.db

Connecting to 'sqlite:///ma_base.db'

In [ ]:
%%sql
SELECT * FROM clients;

Running query in 'sqlite:///ma_base.db'

++
||
++
++

In [11]:
%%sql

SELECT * FROM clients
WHERE [age] >= (SELECT AVG([age]) FROM clients);

/*
# Objectif : introduction aux sous-requêtes
# Jusqu'à maintenant, lorsqu'on utilisait les fonctions (ex : SUM, AVG, SUBSTRING, etc.), on les utilisait sur la ligne du SELECT.
# Chaque fonction créait une nouvelle colonne.

# Mais si l'on souhaite les utiliser pour du filtrage ou des opérations dans les lignes suivantes, certaines de ces fonctions ne peuvent pas être réutilisées dans ce cadre.
# C'est le cas des fonctions d'agrégation (SUM, AVG, COUNT, etc.), qui ne peuvent pas être utilisées dans un WHERE.
# En effet, le WHERE est évalué avant les fonctions d'agrégation (pour décider quelles lignes garder), mais les fonctions d'agrégation s'appuient sur toutes les lignes.
# Donc c'est une référence circulaire, si l'on décide de filtrer (WHERE) selon la valeur de la fonction d'agrégation.

# Si l'on souhaite contourner ce problème, on peut utiliser les sous-requêtes.
# Elles permettent d'assurer ces filtrages et opérations dans les lignes qui suivent le SELECT.
# Par exemple, le code suivant produirait une erreur :
#     SELECT *, AVG([age]) AS Moyenne FROM clients
#     WHERE [age] >= Moyenne;

# Mais en réécrivant ce code avec une sous-requête, il n'y a plus l'erreur et on obtient le résultat voulu :
#     SELECT * FROM clients
#     WHERE [age] >= (SELECT AVG([age]) FROM clients);

# Note : les sous-requêtes peuvent être utilisées dans différentes commandes : WHERE, HAVING, FROM, JOIN (et autres ?).
*/

Running query in 'sqlite:///ma_base.db'

id,nom,email,age,Annee,avg_age_annee
4,Gilbert,gilbert@gmail.com,50,2021,44.666666666666664
5,Romane,romane@gmail.com,50,2021,44.666666666666664
10,Melissa,"melissa@yahoo,fr",56.77,2022,35.81


In [6]:
%%sql
SELECT AVG([age]) FROM clients
WHERE [Annee] = 2020
;

Running query in 'sqlite:///ma_base.db'

AVG([age])
28.333333333333332


In [5]:
%%sql
SELECT [id], [Annee],

CASE
    WHEN [Annee] = 2020 THEN (SELECT AVG([age]) AS avg_age FROM clients WHERE [Annee] = 2020)
    WHEN [Annee] = 2021 THEN (SELECT AVG([age]) AS avg_age FROM clients WHERE [Annee] = 2021)
    WHEN [Annee] = 2022 THEN (SELECT AVG([age]) AS avg_age FROM clients WHERE [Annee] = 2022)
END AS case_test

FROM clients;

Running query in 'sqlite:///ma_base.db'

id,Annee,case_test
1,2020,28.333333333333332
2,2020,28.333333333333332
3,2020,28.333333333333332
4,2021,44.666666666666664
5,2021,44.666666666666664
6,2021,44.666666666666664
7,2022,35.81
8,2022,35.81
9,2022,35.81
10,2022,35.81


In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("ma_base.db")


texte = "SELECT [id], [Annee],\n\nCASE\n"

for i in range(2020, 2022+1,1):
    texte += f"    WHEN [Annee] = {i} THEN (SELECT AVG([age]) AS avg_age FROM clients WHERE [Annee] = {i})\n"

texte += "END AS avg_age\n\n"
texte += "FROM clients;"
    
print(texte)
pd.read_sql(texte, conn)

SELECT [id], [Annee],

CASE
    WHEN [Annee] = 2020 THEN (SELECT AVG([age]) AS avg_age FROM clients WHERE [Annee] = 2020)
    WHEN [Annee] = 2021 THEN (SELECT AVG([age]) AS avg_age FROM clients WHERE [Annee] = 2021)
    WHEN [Annee] = 2022 THEN (SELECT AVG([age]) AS avg_age FROM clients WHERE [Annee] = 2022)
END AS avg_age

FROM clients;


,id,Annee,avg_age
0,1,2020,28.333333
1,2,2020,28.333333
2,3,2020,28.333333
3,4,2021,44.666667
4,5,2021,44.666667
5,6,2021,44.666667
6,7,2022,35.810000
7,8,2022,35.810000
8,9,2022,35.810000
9,10,2022,35.810000


In [2]:
%%sql

SELECT * FROM clients AS c1
WHERE [age] >= (SELECT AVG([age]) FROM clients WHERE c1.Annee = Annee);

Running query in 'sqlite:///ma_base.db'

id,nom,email,age,Annee,avg_age_annee
2,Bob,bob@example.com,35,2020,28.333333333333332
4,Gilbert,gilbert@gmail.com,50,2021,44.666666666666664
5,Romane,romane@gmail.com,50,2021,44.666666666666664
10,Melissa,"melissa@yahoo,fr",56.77,2022,35.81


In [3]:
%%sql
/*
ALTER TABLE clients
ADD avg_age_annee FLOAT;
*/
/*
UPDATE clients AS c1
SET avg_age_annee = (SELECT AVG([age]) FROM clients WHERE c1.Annee = Annee);
*/
SELECT * FROM clients
WHERE [age] >= avg_age_annee;

Running query in 'sqlite:///ma_base.db'

id,nom,email,age,Annee,avg_age_annee
2,Bob,bob@example.com,35,2020,28.333333333333332
4,Gilbert,gilbert@gmail.com,50,2021,44.666666666666664
5,Romane,romane@gmail.com,50,2021,44.666666666666664
10,Melissa,"melissa@yahoo,fr",56.77,2022,35.81
